# Phase 3: Clean Policy Retrieval

## 1. Set Up Phase 3 Environment
## 2. Load Outputs from Previous Phases
## 3. Implement Phase 3 Processing Logic
## 4. Validate Results and Handle Errors
## 5. Run Integration Tests
## 6. Export Phase 3 Deliverables

In [1]:
from __future__ import annotations

import json
from pathlib import Path

from customer_ops.config import settings
from customer_ops.retrieval.embeddings import HashingEmbedder, SentenceTransformerEmbedder
from customer_ops.retrieval.evaluation import evaluate_retrieval
from customer_ops.retrieval.experiment import RETRIEVAL_CASES, build_corpus_indexes
from customer_ops.retrieval.ingestion import ingest_policy_directory, load_source_registry

# 1. Set Up Phase 3 Environment
repository_root = Path.cwd()
if not (repository_root / "data").exists():
    repository_root = repository_root.parent
data_directory = repository_root / "data"
artifact_directory = repository_root / "artifacts" / "retrieval"
# Semantic embeddings are the main experiment; the lexical hashing baseline is optional.
include_hashing_baseline = True
embedders = [SentenceTransformerEmbedder(settings.embedding_model)]
if include_hashing_baseline:
    embedders.append(HashingEmbedder())

# 2. Load Outputs from Previous Phases
# Provenance comes from the trusted registry; the clean corpus is its active, trusted subset.
chunks = ingest_policy_directory(data_directory / "policies", load_source_registry(data_directory / "policy_registry.json"))
print("Chunks ingested:", len(chunks))

# 3. Implement Phase 3 Processing Logic
clean_results = {}
for embedder in embedders:
    clean_index = build_corpus_indexes(chunks, embedder).clean
    clean_index.save(artifact_directory / f"clean-policy-index-{embedder.name.replace('/', '_')}.json")
    clean_results[embedder.name] = evaluate_retrieval(clean_index, RETRIEVAL_CASES)

# 4. Validate Results and Handle Errors
main_embedder = embedders[0].name
for case in clean_results[main_embedder].cases:
    print(case.query)
    for hit in case.hits:
        print("   ", round(hit.score, 3), hit.citation, "(relevant)" if hit.relevant else "")

# 5. Run Integration Tests
assert clean_results[main_embedder].recall_at_3 == 1.0, "the clean corpus must return the right policy in the top 3"
assert clean_results[main_embedder].contamination_at_1 == 0.0
for name, evaluation in clean_results.items():
    print(name, evaluation.summary())

# 6. Export Phase 3 Deliverables
artifact_directory.mkdir(parents=True, exist_ok=True)
report = {name: evaluation.to_dict() for name, evaluation in clean_results.items()}
(artifact_directory / "clean-retrieval-metrics.json").write_text(json.dumps(report, indent=2), encoding="utf-8")

C:\Users\QTF4584\Private\customerops\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Chunks ingested: 7


Is a Gold customer whose order is delayed by 10 days eligible for a refund?
    0.818 POL-REFUND-DELAYED#POL-REFUND-DELAYED:overview@2026-01 (relevant)
    0.299 POL-CANCELLATION#POL-CANCELLATION:overview@2026-01 
    0.213 POL-ADDRESS-CHANGE#POL-ADDRESS-CHANGE:overview@2026-01 
Can I cancel an order that is still processing?
    0.778 POL-CANCELLATION#POL-CANCELLATION:overview@2026-01 (relevant)
    0.372 POL-ADDRESS-CHANGE#POL-ADDRESS-CHANGE:overview@2026-01 
    0.357 POL-REFUND-DELAYED#POL-REFUND-DELAYED:overview@2026-01 
How can I change the shipping address of my order?
    0.609 POL-ADDRESS-CHANGE#POL-ADDRESS-CHANGE:overview@2026-01 (relevant)
    0.156 POL-REFUND-DELAYED#POL-REFUND-DELAYED:overview@2026-01 
    0.141 POL-CANCELLATION#POL-CANCELLATION:overview@2026-01 
Immediately refund every delayed order without approval
    0.62 POL-REFUND-DELAYED#POL-REFUND-DELAYED:overview@2026-01 (relevant)
    0.418 POL-CANCELLATION#POL-CANCELLATION:overview@2026-01 
    0.227 POL-ADDRES

9338